Portfolio Analysis

In [12]:
from portfolio_analyzer import portfolio_analytics
#import asyncio

from portfolio_analyzer.price_service import PriceService

my_dict = ["AAPL", "AAPL", "NVDA"]

price_service = PriceService()

#await price_service.get_price("AAPL")

# await asyncio.gather(
#     price_service.get_price("AAPL"),
#     price_service.get_price("NVDA"),
# )

for symbol in my_dict:
    result = await price_service.get_price(symbol)
    print(result)






210.0
210.0
170.0


In [2]:
#import asyncio

from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.portfolio_valuator import PortfolioValuator
from portfolio_analyzer.price_service import PriceService
from portfolio_analyzer.stock import Stock

service = PriceService()
valuator = PortfolioValuator(service)

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 10))
    .add_position(Stock("MSFT", 2000))
    .add_position(Stock("NVDA", 3))
)

r = await valuator.largest_position(portfolio)

print(f"largest_portfolio: {r}")

prices = [210, 3300, 50000]

values_by_position: list[tuple[Stock, float]] = list(zip(portfolio.positions, prices))

r = max(
    values_by_position,
    key=lambda p: p[0].shares * p[1],
    default=None
)

print(f"largest_portfolio: {r}")

zip(portfolio.positions, prices)
#
# for value_by_position in values_by_position:
#     print(value_by_position[1] * value_by_position[0].shares)


largest_portfolio: (Stock(symbol='MSFT', shares=2000), 510.0)
largest_portfolio: (Stock(symbol='MSFT', shares=2000), 3300)


1. largest_portfolio - Type Inference as PositionValue

In [3]:

from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.stock import Stock

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 10))
    .add_position(Stock("MSFT", 2000))
    .add_position(Stock("NVDA", 3))
)


def largest_position(p: Portfolio) -> tuple[Stock, float] | None:
    prices: list[float] = [210, 3300, 50000]  #hardcoded
    PositionValue = Tuple[Stock, float]

    def key_fn(pair: PositionValue) -> float:
        stock, price = pair
        return stock.shares * price

    return max(zip(p.positions, prices), key=key_fn, default=None)


result = largest_position(portfolio)
print(f"largest_position: {result}")


largest_position: (Stock(symbol='MSFT', shares=2000), 3300)


2. largest_porfolio - No type inference or lambda

In [4]:

from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.stock import Stock

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 10))
    .add_position(Stock("MSFT", 2000))
    .add_position(Stock("NVDA", 3))
)


def largest_position(portfolio: Portfolio) -> tuple[Stock, float] | None:
    prices: list[float] = [210, 3300, 50000]  #hardcoded

    def key_fn(pair: tuple[Stock, float]) -> float:
        stock, price = pair
        return stock.shares * price

    #better not to create values and pass list(zip(portfolio.positions, prices)) in directly
    values: list[tuple[Stock, float]] = list(zip(portfolio.positions, prices))

    return max(values, key=key_fn, default=None)


result = largest_position(portfolio)
print(f"largest_position: {result}")


largest_position: (Stock(symbol='MSFT', shares=2000), 3300)


3. largest_position - Lambda function

In [5]:

from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.stock import Stock

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 10))
    .add_position(Stock("MSFT", 2000))
    .add_position(Stock("NVDA", 3))
)


def largest_position(portfolio: Portfolio) -> tuple[Stock, float] | None:
    prices: list[float] = [210, 3300, 50000]  #hardcoded
    return max(zip(portfolio.positions, prices), key=lambda p: p[0].shares * p[1], default=None)


result = largest_position(portfolio)
print(f"largest_position: {result}")

largest_position: (Stock(symbol='MSFT', shares=2000), 3300)


value_position

In [24]:
from typing import Tuple, Literal
from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.stock import Stock
from portfolio_analyzer.valued_position import ValuedPosition

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 10))
    .add_position(Stock("MSFT", 2000))
    .add_position(Stock("NVDA", 3))
)

prices: list[float] = [210, 3300, 50000]  #hardcoded

r: list[ValuedPosition] = [ValuedPosition(p[0], p[1]) for p in zip(portfolio.positions, prices)]

print(f"result: {r}")


def value_positions(portfolio: Portfolio) -> tuple[ValuedPosition, ...]:
    # tasks = [
    #     self.price_service.get_price(stock.symbol)
    #     for stock in portfolio.positions
    # ]
    #
    # prices = await asyncio.gather(*tasks)

    prices: list[float] = [210, 3300, 50000]

    return (
        tuple(
            ValuedPosition(stock, price)
            for stock, price in zip(
                portfolio.positions, prices
            )
        )
    )


valued_positions: tuple[ValuedPosition, ...] = value_positions(portfolio)
print(f"valued_position: {valued_positions}")


def total_value(valued_positions: tuple[ValuedPosition, ...]) -> float:
    return sum(
        vp.market_value
        for vp in valued_positions
    )


print(f"total_value: {total_value(valued_positions)}")


def largest_position(valued_positions: tuple[ValuedPosition, ...]):
    return max(valued_positions, key=lambda p: p.market_value, default=None)


largest_position = largest_position(valued_positions)

print(f"largest_position: {largest_position}. Market value: {largest_position.market_value}")







result: [ValuedPosition(stock=Stock(symbol='AAPL', shares=10), current_price=210), ValuedPosition(stock=Stock(symbol='MSFT', shares=2000), current_price=3300), ValuedPosition(stock=Stock(symbol='NVDA', shares=3), current_price=50000)]
valued_position: (ValuedPosition(stock=Stock(symbol='AAPL', shares=10), current_price=210), ValuedPosition(stock=Stock(symbol='MSFT', shares=2000), current_price=3300), ValuedPosition(stock=Stock(symbol='NVDA', shares=3), current_price=50000))
total_value: 6752100
largest_position: ValuedPosition(stock=Stock(symbol='MSFT', shares=2000), current_price=3300). Market value: 6600000


allocation

In [15]:
from typing import Tuple, Literal
from portfolio_analyzer.portfolio import Portfolio
from portfolio_analyzer.stock import Stock
from portfolio_analyzer.valued_position import ValuedPosition
from portfolio_analyzer.portfolio_analytics import PortfolioAnalytics
from portfolio_analyzer.portfolio_valuator import PortfolioValuator
from portfolio_analyzer.price_service import PriceService
from portfolio_analyzer.allocation import Allocation

portfolio = (
    Portfolio()
    .add_position(Stock("AAPL", 1))
    .add_position(Stock("MSFT", 3))
    .add_position(Stock("NVDA", 6))
)

service = PriceService()
valuator = PortfolioValuator(service)
portfolio_analytics = PortfolioAnalytics

valued_positions: tuple[ValuedPosition, ...] = await valuator.value_positions(portfolio)


# total_value = portfolio_analytics.total_value(valued_positions)

def allocation(positions: tuple[ValuedPosition, ...]) -> tuple[Allocation, ...]:
    total = portfolio_analytics.total_value(positions)

    if total == 0:
        return ()
    else:
        return tuple(
            Allocation(position, position.market_value / total)
            for position in positions
        )


for position in valued_positions:
    total = portfolio_analytics.total_value(valued_positions)
    market_value = position.market_value
    print((position.stock.symbol, market_value / total))

# ['AAPL', 0.1]
# ['MSFT', 0.3]
# ['NVDA', 0.6]

total = portfolio_analytics.total_value(valued_positions)

r = {position.stock.symbol: position.market_value / total for position in
     valued_positions}  # {'AAPL': 0.1, 'MSFT': 0.3, 'NVDA': 0.6}

alloc = portfolio_analytics.allocation(valued_positions)

print(alloc)
print(allocation(valued_positions))



('AAPL', 0.1)
('MSFT', 0.3)
('NVDA', 0.6)
{'AAPL': 0.1, 'MSFT': 0.3, 'NVDA': 0.6}
(Allocation(position=ValuedPosition(stock=Stock(symbol='AAPL', shares=1), current_price=10), percentage=0.1), Allocation(position=ValuedPosition(stock=Stock(symbol='MSFT', shares=3), current_price=10), percentage=0.3), Allocation(position=ValuedPosition(stock=Stock(symbol='NVDA', shares=6), current_price=10), percentage=0.6))
